# Nivelul 0 — Primii pași în calculul cuantic

*Workshop introductiv · Qiskit Fall Fest Iași 2026*

În seara aceasta veți scrie primele programe pentru un calculator cuantic. Lucrați în ritmul vostru: celulele gri se rulează cu **Shift + Enter**, iar casetele verzi sunt mici exerciții.

1. Python pentru cei care știu C++
2. Primul qubit
3. Rotații
4. Faza
5. Doi qubiți și încâlcirea
6. Zarul cuantic

In [ ]:
# Rulați această celulă prima (Shift + Enter).
import sys
import numpy as np
import matplotlib.pyplot as plt
import qiskit
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit.primitives import StatevectorSampler
from qiskit.visualization import plot_histogram


def masoara(qc, shots=1000):
    """Rulează circuitul de shots ori pe un simulator ideal și returnează numărătorile."""
    c = qc.copy()
    if not any(instr.operation.name == "measure" for instr in c.data):
        c.measure_all()
    rezultat = StatevectorSampler().run([c], shots=shots).result()
    return rezultat[0].data.meas.get_counts()


print(f"Python {sys.version.split()[0]} · Qiskit {qiskit.__version__} — totul e pregătit!")

## 1. Python pentru cei care știu C++

Python seamănă cu C++, dar e mai scurt: nu declarați tipuri, nu puneți `;` și nici acolade. **Blocurile se marchează prin indentare** (4 spații).

| C++ | Python |
| --- | --- |
| `int x = 3;` | `x = 3` |
| `vector<int> v = {1, 2, 3};` | `v = [1, 2, 3]` |
| `v.push_back(4);` | `v.append(4)` |
| `for (int i = 0; i < 5; i++) { ... }` | `for i in range(5):` + bloc indentat |
| `if (x > 2 && y == 0) { ... }` | `if x > 2 and y == 0:` + bloc indentat |
| `int f(int a) { return 2 * a; }` | `def f(a):` + `return 2 * a` |
| `cout << "x = " << x << endl;` | `print(f"x = {x}")` |
| `map<string, int> m; m["a"] = 1;` | `m = {"a": 1}` (dicționar) |
| `#include <cmath>` + `sqrt(2)` | `import numpy as np` + `np.sqrt(2)` |

In [ ]:
patrate = []
for i in range(5):
    patrate.append(i * i)
print(patrate)


def dublu(a):
    return 2 * a


print(f"dublu(21) = {dublu(21)}")

numaratori = {"0": 507, "1": 493}          # exact așa arată rezultatele unui circuit
print("De câte ori a ieșit 1:", numaratori["1"])

<div class="alert alert-block alert-success">
<b>Exercițiul 1</b>

Scrieți o funcție <code>suma_cifrelor(n)</code> care returnează suma cifrelor unui număr natural. Indiciu: <code>n % 10</code> și <code>n // 10</code> merg ca în C++ (<code>//</code> este împărțirea întreagă).
</div>

In [ ]:
def suma_cifrelor(n):
    # scrieți aici
    return 0


print(suma_cifrelor(2026))   # ar trebui să afișeze 10

## 2. Primul qubit

Un qubit pornește în starea $|0\rangle$. Poarta **H** (Hadamard) îl pune în superpoziția $\frac{1}{\sqrt2}(|0\rangle + |1\rangle)$: la măsurare obținem 0 sau 1, fiecare cu probabilitatea 50%.

- `Statevector(qc)` arată amplitudinile (posibil doar în simulare);
- `masoara(qc)` rulează circuitul de 1000 de ori, ca un experiment real.

In [ ]:
qc = QuantumCircuit(1)
qc.h(0)
display(qc.draw("mpl"))

stare = Statevector(qc)
print("Amplitudini:", np.round(stare.data, 3))
print("Probabilități:", {k: round(float(v), 3) for k, v in stare.probabilities_dict().items()})

counts = masoara(qc)
print("1000 de măsurători:", counts)
plot_histogram(counts)

<div class="alert alert-block alert-success">
<b>Exercițiul 2</b>

Construiți un circuit care dă <b>mereu</b> 1 la măsurare. Ce poartă folosiți?
</div>

In [ ]:
qc = QuantumCircuit(1)
# scrieți aici

print(masoara(qc))

## 3. Rotații

Poarta $R_y(\theta)$ rotește qubitul cu unghiul $\theta$ și ne lasă să alegem orice probabilitate între 0 și 1. Rulați celula și urmăriți cum crește probabilitatea lui 1 când crește unghiul.

In [ ]:
unghiuri = np.linspace(0, np.pi, 9)
p1 = []
for theta in unghiuri:
    qc = QuantumCircuit(1)
    qc.ry(theta, 0)
    p1.append(Statevector(qc).probabilities()[1])
    print(f"θ = {theta:.2f} rad   →   P(1) = {p1[-1]:.3f}")

plt.plot(unghiuri, p1, "o-")
plt.xlabel("θ (radiani)")
plt.ylabel("P(1)")
plt.grid(alpha=0.3)
plt.show()

**Întrebare pentru acasă:** ce formulă trigonometrică leagă unghiul θ de probabilitatea lui 1? Uitați-vă la valorile pentru θ = 0, π/2 și π.

## 4. Faza — ce nu se vede la măsurare

$|+\rangle = \frac{1}{\sqrt2}(|0\rangle + |1\rangle)$ și $|-\rangle = \frac{1}{\sqrt2}(|0\rangle - |1\rangle)$ diferă doar printr-un semn minus, adică prin **fază**. Măsurate direct, arată la fel. O poartă H pusă înainte de măsurare le deosebește.

In [ ]:
plus = QuantumCircuit(1)
plus.h(0)

minus = QuantumCircuit(1)
minus.x(0)
minus.h(0)          # X apoi H dă |−⟩

print("|+⟩ măsurat direct:", masoara(plus))
print("|−⟩ măsurat direct:", masoara(minus))

plus.h(0)
minus.h(0)
print("|+⟩ cu încă un H:  ", masoara(plus))
print("|−⟩ cu încă un H:  ", masoara(minus))

## 5. Doi qubiți și încâlcirea

Doi qubiți au patru stări de bază: 00, 01, 10, 11. **Atenție:** Qiskit scrie qubitul 0 **în dreapta**.

Poarta **CNOT**, `qc.cx(control, tinta)`, întoarce qubitul țintă doar dacă qubitul de control este 1. Verificați tabelul de adevăr:

In [ ]:
for intrare in ["00", "01", "10", "11"]:
    qc = QuantumCircuit(2)
    if intrare[1] == "1":   # caracterul din dreapta = qubitul 0
        qc.x(0)
    if intrare[0] == "1":   # caracterul din stânga = qubitul 1
        qc.x(1)
    qc.cx(0, 1)             # control: qubitul 0, țintă: qubitul 1
    iesire = list(masoara(qc, 100))[0]   # un singur rezultat posibil
    print(f"{intrare}  →  {iesire}")

Acum combinăm H cu CNOT și obținem o **pereche Bell**:

$$|\Phi^+\rangle = \frac{1}{\sqrt2}\left(|00\rangle + |11\rangle\right)$$

Fiecare qubit, privit singur, e complet aleator. Dar cei doi sunt mereu de acord: nu apar niciodată 01 sau 10.

In [ ]:
bell = QuantumCircuit(2)
bell.h(0)
bell.cx(0, 1)
display(bell.draw("mpl"))
print("Amplitudini:", np.round(Statevector(bell).data, 3))
plot_histogram(masoara(bell))

## 6. Zarul cuantic

Trei qubiți în superpoziție dau 8 rezultate la fel de probabile: un zar cu 8 fețe. Același circuit l-am trimis la începutul serii pe un calculator cuantic IBM real.

<div class="alert alert-block alert-success">
<b>Exercițiul 3</b>

Construiți zarul: un circuit cu 3 qubiți și o poartă H pe fiecare. Aruncați-l de 8.000 de ori. Ies toate fețele exact de câte 1.000 de ori? De ce?
</div>

In [ ]:
zar = QuantumCircuit(3)
# scrieți aici

plot_histogram(masoara(zar, 8000))

## Pe mâine!

Ați scris primele programe cuantice. Mâine, la 08:30, vă așteaptă ceva mai mare. Lăsați laptopul încărcat și Jupyter instalat exact cum e acum.

**Opțional, pentru curioși:** cursul gratuit *Basics of Quantum Information*, din secțiunea Learning a platformei [IBM Quantum](https://quantum.cloud.ibm.com/).